# 05 · Inspect raw status or confirm one loop/pump operation

Start with `ACTION = "READ"`: this sends queries only and saves three rounds of exact CTL/PMP/LVL replies to a new JSONL file under `OUTPUT_DIR`. Record the physical panel state alongside them. No Stop, Run or pump command is sent. `QUIT` is retained verbatim; Stop accepts S/QUIT and Run requires R.

For control, choose `RUN`, `STOP`, `PUMP ON` or `PUMP OFF` and confirm the selected action at the prompt. Run uses the existing target. The device decides whether it accepts the request; the API verifies readback.

Read [the tutorial guide](README.md) and [physical commissioning](../docs/commissioning.md) first. Use one serial owner. Notebook outputs are empty; no bench result is implied.

## Environment

From the repository root, run:

```powershell
uv sync --locked
uv run --locked --with jupyterlab --with ipykernel python -m jupyterlab
```

Choose that environment's Python kernel and run top to bottom. Restart the kernel after driver changes. The next cell opens no serial port.


In [ ]:
import sys
from pathlib import Path

# Prefer this checkout when launched from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, REPO_ROOT.parent):
    if (candidate / "src/tark_nrc/controller.py").is_file():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
print(f"Python: {sys.executable}")
print(f"Repository/config base: {REPO_ROOT}")

## Imports


In [ ]:
import json
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path
from time import sleep

from tark_nrc import ProtocolError, create_chiller

## Choose connection settings and ONE operation

Set `CONFIG_PATH` to your verified NRC400 profile. This keeps USB adapter/unit identity bindings and verified serial framing/timeouts together. The current `[control]` section accepts only `expected_serial_number`; remove old `allow_writes`, `min_c` and `max_c` entries. The API has no write-enable switch, target range or state-based control lock. Choose targets for the actual coolant and experiment. There is no guessed COM port. `SIMULATE = True` ignores the hardware profile and opens no physical port; each run creates a fresh simulated device with no shared state. Review all operation settings before continuing.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config.toml"
SIMULATE = False
ACTION = "READ"  # READ sends queries only. Controls: RUN, STOP, PUMP ON or PUMP OFF.
OUTPUT_DIR = REPO_ROOT / "outputs"

## Connect, inspect, confirm one operation, verify and release

Run this entire operation cell together: connection and cleanup share one context/finally block. Ctrl+C/Interrupt attempts cleanup while allowing Python to finish. Do not split the cell, kill the kernel during I/O, or rerun setup over an unfinished connection. Errors remain visible. Closing the port retains the equipment's target, loop and pump state.


In [ ]:
if ACTION not in ("READ", "RUN", "STOP", "PUMP ON", "PUMP OFF"):
    raise ValueError("ACTION must be READ, RUN, STOP, PUMP ON or PUMP OFF")

chiller = create_chiller(None if SIMULATE else CONFIG_PATH, simulate=SIMULATE)
with chiller:
    print(f"Backend: {chiller.transport.backend}; identity: {chiller.identity}")
    before = chiller.read_sample()
    print(f"Target: {before.setpoint_c:.3f} C; supply: {before.temperature_c:.2f} C")
    print(f"Loop: {before.control_loop}; pump: {before.pump}; level: {before.level}")
    if ACTION == "READ":
        # Preserve exact replies, including the firmware's QUIT status.
        stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        path = OUTPUT_DIR / f"{chiller.transport.backend}_loop_status_{stamp}.jsonl"
        print(f"Read-only diagnostic; no control commands. Saving to: {path.resolve()}")
        with path.open("x", encoding="utf-8") as evidence:
            metadata = {
                "backend": chiller.transport.backend,
                "identity": asdict(chiller.identity),
                "port": chiller.transport.port,
                "sample": before.to_dict(),
            }
            evidence.write(json.dumps(metadata) + "\n")
            evidence.flush()
            for index in range(3):
                reading = {"timestamp_utc": datetime.now(UTC).isoformat()}
                for code in ("CTL", "PMP", "LVL"):
                    reading[code] = chiller.query(code)  # Query only: no comma/data field.
                print(f"Raw replies {index + 1}: {reading!r}")
                evidence.write(json.dumps(reading) + "\n")
                evidence.flush()
                if index < 2:
                    sleep(0.5)
        print("Note the front-panel loop/pump state alongside these raw serial replies.")
    else:
        print("RUN uses the existing target. Prepare the primed loop and inspect panel alarms.")
        print("STOP requests regulation stop; read the pump state separately.")
        print("The device decides whether it accepts each command; the API verifies readback.")
        print("Check heat-load readiness before reducing cooling or circulation.")
        if input(f"Type {ACTION!r} to send ONE request; anything else cancels: ").strip() == ACTION:
            if ACTION == "RUN":
                chiller.run()  # Sends CTL,R once and verifies reported R.
            elif ACTION == "STOP":
                chiller.stop()  # Sends CTL,S once; bounded polling never repeats the write.
            else:
                chiller.set_pump(ACTION == "PUMP ON")  # Sends PMP once and verifies ON/OFF.
            after = chiller.read_sample()
            if ACTION == "STOP":
                matches = after.control_loop in ("S", "QUIT")
            elif ACTION == "RUN":
                matches = after.control_loop == "R"
            else:
                matches = after.pump == ("ON" if ACTION == "PUMP ON" else "OFF")
            if not matches:
                raise ProtocolError(
                    "State changed after verification; inspect the panel. Do not retry."
                )
            print(f"Reported loop: {after.control_loop}; pump: {after.pump}; level: {after.level}")
            print("Compare with the panel and observe flow/temperature separately.")
        else:
            print("Cancelled; no control request sent.")

print("Connection closed. Equipment retains its state; no automatic stop or rollback.")

## Check results and finish

For READ, keep the printed raw replies and the JSONL file, and record what the physical panel shows. If diagnosing the prior Stop failure, restart the kernel before loading the updated driver error messages.


Compare reported state with the panel, flow and temperature. Stop does not guarantee pump-off; use a separate confirmed pump-off operation when appropriate. Manage the heat load before reducing cooling/circulation. An uncertain result requires equipment inspection before another request; there is no retry, automatic shutdown sequence or rollback.

**Software connection closure does not stop the chiller or pump.** Follow the physical shutdown procedure separately and confirm panel state. A forced kernel termination cannot guarantee cleanup. See [the hardware test record checklist](README.md#6-record-a-hardware-test-result) for evidence to retain.
